In [ ]:
# [1] Setup TPU v5e Environment & Extract Standalone Code
import os, sys, base64
os.environ['PJRT_DEVICE'] = 'TPU'
print('PJRT_DEVICE =', os.environ.get('PJRT_DEVICE'))
try:
    import torch, torch_xla
    print('torch:', torch.__version__)
    print('torch_xla import OK (device query deferred: training workers own the single TPU client).')
except Exception as e:
    print('torch_xla import note (installs on demand):', e)
    os.system('pip install -q torch_xla')
CODE_B64 = """..."""
with open('/kaggle/working/dioptra_dino.py', 'wb') as f:
    f.write(base64.b64decode(CODE_B64))
src = open('/kaggle/working/dioptra_dino.py').read()
assert 'tpu_static_loss' in src and 'xm.optimizer_step' in src, 'TPU symbols missing!'
assert 'if domain.startswith("tartan") and H == W:' in src, 'Reason-A fix missing!'
print('TPU v5e Codebase Deployed with Reason-A + XLA verified.')


In [ ]:
# [2] Pre-index datasets ONCE + warm DINO weights (avoids 8-worker FUSE stampede & TPU idle kill)
# Spawned workers reuse /kaggle/working/dataset_index_train.json in ~0.05s instead of
# each walking 100GB+ of FUSE mounts while the TPU sits idle (idle >2h => exit 137).
import os, sys, time
if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')
t0 = time.time()
from dioptra_dino import MultiDomainDINODataset, DINOV2_VITS14_URL
ds = MultiDomainDINODataset(root_dirs='auto', split='train', image_size=336, apply_pinhole_aug=True, crop_min=0.35)
print(f'Pre-indexed {len(ds):,} train samples in {time.time()-t0:.1f}s -> cache warmed.')
del ds
for p in ('/kaggle/working/dataset_index_train.json', '/kaggle/working/dataset_index_train.pkl'):
    print(p, 'size:', os.path.getsize(p) if os.path.exists(p) else 'MISSING')
assert any(os.path.exists(p) and os.path.getsize(p) > 1024 for p in ('/kaggle/working/dataset_index_train.json', '/kaggle/working/dataset_index_train.pkl')), 'index cache save failed!'
print('Index cache verified on disk.')
t0 = time.time()
try:
    import torch
    torch.hub.load_state_dict_from_url(DINOV2_VITS14_URL, map_location='cpu')
    print(f'DINOv2 weights cached in {time.time()-t0:.1f}s.')
except Exception as e:
    print('DINO weights warmup note:', e)


In [ ]:
# [3] Launch 8-Core Distributed Training via PyTorch/XLA (inner xmp.spawn, all devices)
import os
os.environ['PJRT_DEVICE'] = 'TPU'
# Foundation checkpoint: direct paths only (NO recursive glob over 100GB+ FUSE mounts).
ckpt_cands = [
    '/kaggle/input/dioptra-dino-epoch5-ckpt/checkpoint_step_latest.pt',
    '/kaggle/input/datasets/yumnamharryson/dioptra-dino-epoch5-ckpt/checkpoint_step_latest.pt',
]
ckpt = next((p for p in ckpt_cands if os.path.exists(p)), 'auto')
print('using ckpt:', ckpt)
print('mounted inputs:', sorted(os.listdir('/kaggle/input'))[:30])
cmd = (
    'python -u /kaggle/working/dioptra_dino.py '
    '--train "auto" '
    '--epochs 10 '
    '--image-size 336 '
    '--batch-size 8 '
    '--grad-accum 4 '
    '--lr-head 5e-5 '
    '--freeze-backbone '
    f'--resume "{ckpt}" '
    '--finetune '
    '--domain-balanced '
    '--sensor-noise '
    '--subset-fraction 0.15 '
    '--tpu '
    '--use-bfloat16 '
    '--tpu-num-cores 8 '
    '--num-workers 4 '
    '--save-interval 500 '
    '--output-dir /kaggle/working/outputs_dino'
)
print(cmd)
exit_code = os.system(cmd)
assert exit_code == 0, f'TPU training failed with exit code {exit_code}'


In [ ]:
# [4] Package Checkpoints
import os
os.system('cd /kaggle/working && zip -q -r dioptra_dino_tpuv5e_checkpoints.zip outputs_dino/')
print('Packaged checkpoints successfully.')
